In [4]:
import os
import glob
import numpy as np
import librosa
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.layers import BatchNormalization, Dropout
from tensorflow.keras.callbacks import ReduceLROnPlateau
import xml.etree.ElementTree as ET
from sklearn.metrics import classification_report


# Function to parse annotations from RML files
def parse_annotations(rml_file_path):
    annotations = []
    try:
        tree = ET.parse(rml_file_path)
        root = tree.getroot()
        for event in root.findall(".//ns0:Event", namespaces={"ns0": "http://www.respironics.com/PatientStudy.xsd"}):
            family = event.attrib.get("Family")
            type_ = event.attrib.get("Type")
            start = float(event.attrib.get("Start"))
            duration = float(event.attrib.get("Duration"))
            end = start + duration
            if family == "Respiratory":  # Focus on apnea annotations
                annotations.append({"type": type_, "start": start, "end": end})
    except Exception as e:
        print(f"Error parsing {rml_file_path}: {e}")
    return annotations


# Function to create spectrograms and labels based on annotations
def create_spectrograms(wav_file_path, annotations, sr=22050, n_mels=128, hop_length=512, target_frames=100):
    spectrograms = []
    labels = []

    try:
        audio, _ = librosa.load(wav_file_path, sr=sr)
        mel_spec = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=n_mels, hop_length=hop_length)
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

        for annotation in annotations:
            start_frame = int(annotation["start"] * sr / hop_length)
            end_frame = int(annotation["end"] * sr / hop_length)

            if end_frame > mel_spec_db.shape[1]:
                continue  # Skip annotations outside the spectrogram's range

            segment = mel_spec_db[:, start_frame:end_frame]

            # Pad or truncate to target_frames
            if segment.shape[1] < target_frames:
                padding = target_frames - segment.shape[1]
                segment = np.pad(segment, ((0, 0), (0, padding)), mode="constant")
            elif segment.shape[1] > target_frames:
                segment = segment[:, :target_frames]

            spectrograms.append(segment)
            labels.append(annotation["type"])
    except Exception as e:
        print(f"Error processing file {wav_file_path}: {e}")

    return np.array(spectrograms), labels


# Function to normalize spectrograms
def normalize_spectrograms(spectrograms):
    spectrograms = np.array(spectrograms)
    spectrograms -= np.min(spectrograms)
    spectrograms /= np.max(spectrograms)
    return spectrograms


# Model definition
def build_model(input_shape, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
        BatchNormalization(),
        tf.keras.layers.MaxPooling2D((2, 2)),
        Dropout(0.3),

        tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
        BatchNormalization(),
        tf.keras.layers.MaxPooling2D((2, 2)),
        Dropout(0.3),

        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        Dropout(0.5),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    return model


# Main function to process folders and train model
def process_and_train_model(wav_folder_path, rml_folder_path, sr=22050, n_mels=128, hop_length=512, target_frames=100):
    all_spectrograms = []
    all_labels = []

    # List files
    rml_files = glob.glob(os.path.join(rml_folder_path, "*.rml"))
    wav_files = glob.glob(os.path.join(wav_folder_path, "*.wav"))

    for wav_file_path in wav_files:
        # Match RML file by shared identifier
        wav_base = os.path.basename(wav_file_path).split("_")[0]
        matching_rml_file = None
        for rml_file_path in rml_files:
            if wav_base in os.path.basename(rml_file_path):
                matching_rml_file = rml_file_path
                break

        if not matching_rml_file:
            print(f"No matching RML file for {wav_file_path}")
            continue

        print(f"Using annotations from {matching_rml_file} for .wav file: {wav_file_path}")
        annotations = parse_annotations(matching_rml_file)
        spectrograms, labels = create_spectrograms(wav_file_path, annotations, sr, n_mels, hop_length, target_frames)
        if spectrograms.size > 0:
            all_spectrograms.extend(spectrograms)
            all_labels.extend(labels)

    if not all_spectrograms or not all_labels:
        print("No data available for training. Check the file paths and ensure annotations exist in the RML files.")
        return

    # Normalize and encode labels
    all_spectrograms = normalize_spectrograms(all_spectrograms)
    label_encoder = LabelEncoder()
    all_labels_encoded = label_encoder.fit_transform(all_labels)

    X_train, X_test, y_train, y_test = train_test_split(all_spectrograms, all_labels_encoded, test_size=0.2, random_state=42)
    X_train = X_train[..., np.newaxis]
    X_test = X_test[..., np.newaxis]

    model = build_model((n_mels, target_frames, 1), len(label_encoder.classes_))
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)

    model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=20, batch_size=32, callbacks=[lr_scheduler])
    loss, accuracy = model.evaluate(X_test, y_test)
    print(f"Test Loss: {loss}, Test Accuracy: {accuracy}")

    # Classification report
    y_pred = model.predict(X_test)
    y_pred_classes = np.argmax(y_pred, axis=1)
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred_classes, target_names=label_encoder.classes_))


# Specify folder paths
wav_folder_path = "/Users/terlan/Library/CloudStorage/OneDrive-uni-mannheim.de/cleaned_wav/Mic"
rml_folder_path = "/Users/terlan/Library/CloudStorage/OneDrive-uni-mannheim.de/cleaned_label"

# Execute
process_and_train_model(wav_folder_path, rml_folder_path)


Using annotations from /Users/terlan/Library/CloudStorage/OneDrive-uni-mannheim.de/cleaned_label/filtered_00001122-100507.rml for .wav file: /Users/terlan/Library/CloudStorage/OneDrive-uni-mannheim.de/cleaned_wav/Mic/00001122-100507_mic_cleaned.wav
Using annotations from /Users/terlan/Library/CloudStorage/OneDrive-uni-mannheim.de/cleaned_label/filtered_00001024-100507.rml for .wav file: /Users/terlan/Library/CloudStorage/OneDrive-uni-mannheim.de/cleaned_wav/Mic/00001024-100507_mic_cleaned.wav
Using annotations from /Users/terlan/Library/CloudStorage/OneDrive-uni-mannheim.de/cleaned_label/filtered_00001018-100507.rml for .wav file: /Users/terlan/Library/CloudStorage/OneDrive-uni-mannheim.de/cleaned_wav/Mic/00001018-100507_mic_cleaned.wav
Using annotations from /Users/terlan/Library/CloudStorage/OneDrive-uni-mannheim.de/cleaned_label/filtered_00001097-100507.rml for .wav file: /Users/terlan/Library/CloudStorage/OneDrive-uni-mannheim.de/cleaned_wav/Mic/00001097-100507_mic_cleaned.wav
Usin

2024-11-19 04:35:01.357996: W tensorflow/core/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


298/298 [==============================] - 10s 31ms/step - loss: inf - accuracy: 0.3896 - val_loss: 19135910005368104763053550872821760.0000 - val_accuracy: 0.2201 - lr: 0.0010
Epoch 2/20
298/298 [==============================] - 8s 27ms/step - loss: nan - accuracy: 0.0403 - val_loss: nan - val_accuracy: 0.0386 - lr: 0.0010
Epoch 3/20
298/298 [==============================] - 8s 27ms/step - loss: nan - accuracy: 0.0396 - val_loss: nan - val_accuracy: 0.0386 - lr: 0.0010
Epoch 4/20
296/298 [============================>.] - ETA: 0s - loss: nan - accuracy: 0.0396
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
298/298 [==============================] - 8s 27ms/step - loss: nan - accuracy: 0.0396 - val_loss: nan - val_accuracy: 0.0386 - lr: 0.0010
Epoch 5/20
298/298 [==============================] - 8s 27ms/step - loss: nan - accuracy: 0.0396 - val_loss: nan - val_accuracy: 0.0386 - lr: 5.0000e-04
Epoch 6/20
298/298 [==============================] - 8s 27ms

/Users/terlan/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1327: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/terlan/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1327: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/terlan/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1327: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
